# DINO dense degradation sweep: checkpoint 0 to 215

This Colab notebook evaluates saved DINO ViT-S/16 checkpoints every 10 epochs from 0 to 210, plus 215. It follows the dense degradation protocol used in *Exploring Structural Degradation in Dense Representations for Self-supervised Learning*: frozen backbone, projector removed, last-layer patch embeddings, lightweight linear semantic segmentation head, PASCAL VOC mIoU curve. It also adds patch-level diagnostics requested during supervision: DSE-style class separability/effective rank, patch feature magnitude histograms, CLS-to-patch attention maps, CLS similarity maps, and PCA patch-feature maps.

Update the paths in the configuration cell before running.

## 1. Mount Google Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf /content/dino
!git clone https://github.com/xbz123/dino-dense-degradation.git /content/dino
%cd /content/dino
!git rev-parse HEAD

## 2. Configure paths and evaluation settings

In [ ]:
from pathlib import Path

# Directory in Google Drive containing checkpoint0000.pth, checkpoint0010.pth, ..., checkpoint0210.pth,
# and either checkpoint0215.pth/checkpoint215.pth or a final checkpoint.pth whose internal epoch is 215/216.
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/dino_checkpoints'

# ImageNet-style image folder used for DSE/patch diagnostics. This should match the pretraining data distribution.
# It must be readable by torchvision.datasets.ImageFolder: root/class_name/image.jpg.
DSE_IMAGE_ROOT = '/content/drive/MyDrive/ImageNet100/train'

OUTPUT_ROOT = '/content/drive/MyDrive/dino_dense_degradation_eval'
WORK_CKPT_DIR = '/content/dino_eval_checkpoints'

TARGET_EPOCHS = list(range(0, 211, 10)) + [215]

# Paper-aligned settings. If Colab runs out of memory, reduce VOC_LINEAR_BATCH_SIZE to 64 or 32.
VOC_IMG_SIZE = 336
VOC_LINEAR_BATCH_SIZE = 128
VOC_LINEAR_LR = 0.01 * VOC_LINEAR_BATCH_SIZE / 256
VOC_LINEAR_EPOCHS = 15

# Paper uses 2048 sampled pretraining images for DSE. Reduce to 256 for a quick smoke test.
NUM_DSE_IMAGES = 2048
NUM_VIS_IMAGES = 6

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
Path(WORK_CKPT_DIR).mkdir(parents=True, exist_ok=True)
print('checkpoint dir:', DRIVE_CHECKPOINT_DIR)
print('DSE image root:', DSE_IMAGE_ROOT)
print('target epochs:', TARGET_EPOCHS)
print('VOC lr:', VOC_LINEAR_LR)

## 3. Copy and verify checkpoints from Google Drive

The evaluator expects epoch numbers in filenames. This cell normalizes all selected checkpoints to `checkpoint####.pth` under `/content/dino_eval_checkpoints`.

In [ ]:
import os, shutil, torch

assert os.path.isdir(DRIVE_CHECKPOINT_DIR), DRIVE_CHECKPOINT_DIR
print('=== Drive checkpoint files ===')
print('\n'.join(sorted(os.listdir(DRIVE_CHECKPOINT_DIR))[:300]))

def read_internal_epoch(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    return ckpt.get('epoch', None) if isinstance(ckpt, dict) else None

def find_checkpoint_for_epoch(epoch):
    candidates = [
        f'checkpoint{epoch:04d}.pth',
        f'checkpoint{epoch:03d}.pth',
        f'checkpoint{epoch}.pth',
    ]
    for name in candidates:
        path = os.path.join(DRIVE_CHECKPOINT_DIR, name)
        if os.path.exists(path):
            return path

    if epoch == 215:
        path = os.path.join(DRIVE_CHECKPOINT_DIR, 'checkpoint.pth')
        if os.path.exists(path):
            internal = read_internal_epoch(path)
            print('fallback checkpoint.pth internal epoch:', internal)
            if internal in (215, 216):
                return path
    return None

prepared = []
for epoch in TARGET_EPOCHS:
    src = find_checkpoint_for_epoch(epoch)
    assert src is not None, f'Missing checkpoint for epoch {epoch} in {DRIVE_CHECKPOINT_DIR}'
    dst = os.path.join(WORK_CKPT_DIR, f'checkpoint{epoch:04d}.pth')
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1024 / 1024
    print(f'{epoch:>4}: {src} -> {dst} | internal_epoch={read_internal_epoch(dst)} | {size_mb:.1f} MB')
    prepared.append(dst)

print('=== Prepared checkpoints ===')
print('\n'.join(sorted(os.listdir(WORK_CKPT_DIR))))

## 4. Patch the VOC linear evaluator to use the paper-style Adam optimizer

In [ ]:
eval_path = '/content/dino/eval_voc_dense.py'
txt = Path(eval_path).read_text()
old = "optimizer = torch.optim.SGD(head.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)"
new = "optimizer = torch.optim.Adam(head.parameters(), lr=lr)"
if old in txt:
    txt = txt.replace(old, new)
    Path(eval_path).write_text(txt)
    print('Patched optimizer: SGD -> Adam')
else:
    print('Optimizer patch was already applied or source changed.')

## 5. Run PASCAL VOC frozen-backbone linear segmentation

This is the main dense degradation curve. It can take a long time because it trains a linear head for every checkpoint.

In [ ]:
!python /content/dino/eval_voc_dense.py \
  --ckpt_dir {WORK_CKPT_DIR} \
  --voc_root /content/voc_data \
  --arch vit_small \
  --patch_size 16 \
  --img_size {VOC_IMG_SIZE} \
  --train_epochs {VOC_LINEAR_EPOCHS} \
  --lr {VOC_LINEAR_LR} \
  --batch_size {VOC_LINEAR_BATCH_SIZE} \
  --feature_dtype float16 \
  --output_dir {OUTPUT_ROOT}/voc_0_to_215

## 6. Write the patch/DSE/attention diagnostic script

In [ ]:
%%writefile /content/dino/patch_attention_dse_suite.py
import os, re, csv, json, math, argparse
from collections import OrderedDict
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
import vision_transformer as vits

def parse_epoch(path):
    match = re.search(r'checkpoint0*(\d+)\.pth', os.path.basename(path))
    return int(match.group(1)) if match else -1

def discover_checkpoints(ckpt_dir):
    files = []
    for name in os.listdir(ckpt_dir):
        if name.endswith('.pth') and parse_epoch(name) >= 0:
            files.append((parse_epoch(name), os.path.join(ckpt_dir, name)))
    files.sort()
    if not files:
        raise FileNotFoundError(f'No checkpoint####.pth files found in {ckpt_dir}')
    return files

def clean_state_dict(state_dict):
    out = OrderedDict()
    for key, value in state_dict.items():
        key = key.replace('module.', '').replace('backbone.', '')
        if key.startswith('head.') or key.startswith('dino_head.'):
            continue
        out[key] = value
    return out

def load_backbone(path, device):
    model = vits.vit_small(patch_size=16, num_classes=0)
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    if isinstance(checkpoint, dict) and 'teacher' in checkpoint:
        state_dict, source = checkpoint['teacher'], 'teacher'
    elif isinstance(checkpoint, dict) and 'student' in checkpoint:
        state_dict, source = checkpoint['student'], 'student'
    else:
        state_dict, source = checkpoint, 'raw'
    msg = model.load_state_dict(clean_state_dict(state_dict), strict=False)
    model.to(device).eval()
    for param in model.parameters():
        param.requires_grad = False
    internal = checkpoint.get('epoch', None) if isinstance(checkpoint, dict) else None
    print(f'loaded {os.path.basename(path)} source={source} internal_epoch={internal}')
    print(f'  missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)}')
    return model, internal

def sample_rows(x, max_rows):
    if x.shape[0] <= max_rows:
        return x
    idx = torch.randperm(x.shape[0], device=x.device)[:max_rows]
    return x[idx]

def effective_rank(x, max_tokens=30000):
    x = sample_rows(x.float(), max_tokens)
    x = x - x.mean(dim=0, keepdim=True)
    singular = torch.linalg.svdvals(x)
    prob = singular / singular.sum().clamp_min(1e-12)
    return torch.exp(-(prob * (prob + 1e-12).log()).sum()).item()

def kmeans(x, k, iters=20):
    n = x.shape[0]
    centers = x[torch.randperm(n, device=x.device)[:k]].clone()
    for _ in range(iters):
        labels = torch.cdist(x, centers).argmin(dim=1)
        new_centers = []
        for i in range(k):
            points = x[labels == i]
            new_centers.append(points.mean(dim=0) if points.numel() else centers[i])
        new_centers = torch.stack(new_centers, dim=0)
        if torch.allclose(new_centers, centers, atol=1e-4):
            break
        centers = new_centers
    return labels, centers

def class_separability(tokens, k, max_tokens=12000):
    x = tokens.reshape(-1, tokens.shape[-1]).float()
    x = sample_rows(x, max_tokens)
    x = F.normalize(x, dim=-1)
    if x.shape[0] <= k:
        return float('nan'), float('nan'), float('nan')
    labels, centers = kmeans(x, k)
    intra, inter = [], []
    for j in range(k):
        points = x[labels == j]
        if points.shape[0] < 2:
            continue
        centered = points - points.mean(dim=0, keepdim=True)
        singular = torch.linalg.svdvals(centered)
        intra.append((singular.sum() / math.sqrt(max(1, points.shape[0] - 1))).item())
        other = torch.cat([centers[:j], centers[j + 1:]], dim=0)
        inter.append(torch.cdist(points, other).min(dim=1).values.mean().item())
    if not intra or not inter:
        return float('nan'), float('nan'), float('nan')
    mintra = float(np.mean(intra))
    minter = float(np.mean(inter))
    return mintra, minter, minter - mintra

def normalize01(array):
    array = np.asarray(array, dtype=np.float32)
    return (array - array.min()) / (np.ptp(array) + 1e-6)

def pca_rgb(patches, h, w):
    x = patches.float() - patches.float().mean(dim=0, keepdim=True)
    try:
        _, _, v = torch.pca_lowrank(x, q=3)
        y = x @ v[:, :3]
    except Exception:
        _, _, vh = torch.linalg.svd(x, full_matrices=False)
        y = x @ vh[:3].T
    y = y.reshape(h, w, 3).cpu().numpy()
    return (y - y.min(axis=(0, 1), keepdims=True)) / (np.ptp(y, axis=(0, 1), keepdims=True) + 1e-6)

def save_overlay(path, image, heat, title, cmap='magma'):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(image)
    ax.imshow(normalize01(heat), cmap=cmap, alpha=0.45, extent=(0, image.size[0], image.size[1], 0))
    ax.set_title(title)
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

def save_hist(path, values, title, xlabel):
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(np.asarray(values, dtype=np.float32).reshape(-1), bins=60)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('count')
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

def write_csv(path, rows):
    keys = list(rows[0].keys())
    with open(path, 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)

@torch.no_grad()
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--ckpt_dir', required=True)
    parser.add_argument('--image_root', required=True)
    parser.add_argument('--out', required=True)
    parser.add_argument('--num_metric_images', type=int, default=2048)
    parser.add_argument('--num_vis_images', type=int, default=6)
    parser.add_argument('--seed', type=int, default=0)
    args = parser.parse_args()

    os.makedirs(args.out, exist_ok=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dataset = datasets.ImageFolder(args.image_root)
    rng = np.random.default_rng(args.seed)
    metric_indices = sorted(rng.choice(len(dataset), size=min(args.num_metric_images, len(dataset)), replace=False).tolist())
    vis_indices = set(metric_indices[:args.num_vis_images])
    print('metric images:', len(metric_indices))
    print('visualized indices:', sorted(vis_indices))

    model_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])
    raw_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

    rows = []
    for epoch, ckpt_path in discover_checkpoints(args.ckpt_dir):
        model, internal_epoch = load_backbone(ckpt_path, device)
        epoch_dir = os.path.join(args.out, f'epoch_{epoch:04d}')
        os.makedirs(epoch_dir, exist_ok=True)
        all_patches, all_norms, all_attn, all_cos = [], [], [], []

        for rank, index in enumerate(metric_indices):
            image_path, label = dataset.samples[index]
            image = Image.open(image_path).convert('RGB')
            raw = raw_transform(image)
            x = model_transform(image).unsqueeze(0).to(device)
            tokens = model.get_intermediate_layers(x, n=1)[0]
            cls = tokens[0, 0]
            patches = tokens[0, 1:]
            h = w = int(math.sqrt(patches.shape[0]))
            attentions = model.get_last_selfattention(x)[0, :, 0, 1:].reshape(-1, h, w)
            attention_mean = attentions.mean(dim=0)
            patch_norm = patches.norm(dim=-1)
            cls_cos = F.normalize(patches, dim=-1) @ F.normalize(cls, dim=-1)

            all_patches.append(patches.cpu())
            all_norms.append(patch_norm.cpu())
            all_attn.append(attention_mean.reshape(-1).cpu())
            all_cos.append(cls_cos.cpu())

            if index in vis_indices:
                tag = f'img{rank:04d}_idx{index}_label{label}'
                raw.save(os.path.join(epoch_dir, f'{tag}_original.png'))
                save_overlay(os.path.join(epoch_dir, f'{tag}_cls_attention.png'), raw, attention_mean.cpu(), 'CLS attention')
                save_overlay(os.path.join(epoch_dir, f'{tag}_patch_norm.png'), raw, patch_norm.reshape(h, w).cpu(), 'patch feature norm', 'plasma')
                save_overlay(os.path.join(epoch_dir, f'{tag}_cls_similarity.png'), raw, cls_cos.reshape(h, w).cpu(), 'patch cosine to CLS', 'viridis')
                plt.imsave(os.path.join(epoch_dir, f'{tag}_pca_patch_features.png'), pca_rgb(patches.cpu(), h, w))
                save_hist(os.path.join(epoch_dir, f'{tag}_patch_norm_hist.png'), patch_norm.cpu(), 'patch feature magnitude', 'L2 norm')
                save_hist(os.path.join(epoch_dir, f'{tag}_cls_attention_hist.png'), attention_mean.cpu(), 'CLS attention magnitude', 'attention')

        token_tensor = torch.stack(all_patches, dim=0)
        flat_tokens = token_tensor.reshape(-1, token_tensor.shape[-1])
        norms = torch.cat(all_norms)
        attn = torch.cat(all_attn)
        cos = torch.cat(all_cos)

        b1_values = [class_separability(token_tensor[i:i + 1], k=3) for i in range(token_tensor.shape[0])]
        b1_mintra, b1_minter, b1_sep = np.nanmean(np.asarray(b1_values, dtype=float), axis=0)
        b8_values = [class_separability(token_tensor[i:i + 8], k=24) for i in range(0, max(0, token_tensor.shape[0] - 7), 8)]
        b8_mintra, b8_minter, b8_sep = np.nanmean(np.asarray(b8_values, dtype=float), axis=0)

        row = {
            'epoch': epoch,
            'internal_epoch': internal_epoch,
            'num_metric_images': len(metric_indices),
            'mdim_erank': effective_rank(flat_tokens),
            'mintra_b1k3': float(b1_mintra),
            'minter_b1k3': float(b1_minter),
            'class_sep_b1k3': float(b1_sep),
            'mintra_b8k24': float(b8_mintra),
            'minter_b8k24': float(b8_minter),
            'class_sep_b8k24': float(b8_sep),
            'class_sep_avg': float(np.nanmean([b1_sep, b8_sep])),
            'patch_norm_mean': norms.mean().item(),
            'patch_norm_std': norms.std().item(),
            'cls_attention_mean': attn.mean().item(),
            'cls_attention_std': attn.std().item(),
            'cls_patch_cos_mean': cos.mean().item(),
            'cls_patch_cos_std': cos.std().item(),
        }
        print(json.dumps(row, indent=2))
        rows.append(row)

    sep = np.asarray([r['class_sep_avg'] for r in rows], dtype=float)
    mdim = np.asarray([r['mdim_erank'] for r in rows], dtype=float)
    lam = float(np.nanstd(sep) / (np.nanstd(mdim) + 1e-12))
    for row in rows:
        row['dse_lambda'] = lam
        row['dse'] = row['class_sep_avg'] + lam * row['mdim_erank']

    write_csv(os.path.join(args.out, 'patch_attention_dse_summary.csv'), rows)
    with open(os.path.join(args.out, 'patch_attention_dse_summary.json'), 'w') as handle:
        json.dump(rows, handle, indent=2)

    metrics = ['dse', 'class_sep_avg', 'mdim_erank', 'patch_norm_mean', 'cls_attention_mean', 'cls_patch_cos_mean']
    xs = [row['epoch'] for row in rows]
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    for axis, metric in zip(axes.reshape(-1), metrics):
        axis.plot(xs, [row[metric] for row in rows], marker='o')
        axis.set_title(metric)
        axis.set_xlabel('checkpoint epoch')
        axis.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(args.out, 'dse_patch_attention_curves.png'), dpi=180)
    plt.close(fig)
    print('saved:', args.out)

if __name__ == '__main__':
    main()


## 7. Run DSE, patch statistics, and CLS attention visualizations

In [ ]:
!python /content/dino/patch_attention_dse_suite.py \
  --ckpt_dir {WORK_CKPT_DIR} \
  --image_root {DSE_IMAGE_ROOT} \
  --out {OUTPUT_ROOT}/patch_attention_dse_0_to_215 \
  --num_metric_images {NUM_DSE_IMAGES} \
  --num_vis_images {NUM_VIS_IMAGES} \
  --seed 0

## 8. Inspect outputs

In [ ]:
!echo '=== VOC results ==='
!ls -lh {OUTPUT_ROOT}/voc_0_to_215 || true
!cat {OUTPUT_ROOT}/voc_0_to_215/voc_miou_results.json || true

!echo '=== DSE / patch / attention summary ==='
!ls -lh {OUTPUT_ROOT}/patch_attention_dse_0_to_215 || true
!cat {OUTPUT_ROOT}/patch_attention_dse_0_to_215/patch_attention_dse_summary.json || true

!echo '=== sample figures ==='
!find {OUTPUT_ROOT}/patch_attention_dse_0_to_215 -maxdepth 2 -type f | head -80 || true